In [1]:
import pandas as pd

raw_folder = r'C:\Users\sahil\projects\boston-mbta-ridership\data\raw\MBTA_Bus_Ridership_by_Trip_Season_Route_Line_and_Stop'
processed  = r'C:\Users\sahil\projects\boston-mbta-ridership\data\processed'

fare_free_routes = ['23', '28', '29']

# Map every year's column names to one standard set
rename_map = {
    'GTFS route_id': 'route_id', 'GTFS direction_id': 'direction_id',
    'Stop Name': 'stop_name', 'GTFS stop_id': 'stop_id', 'Year': 'year',
    'Day Type': 'day_type', 'Boardings': 'boardings', 'Alightings': 'alightings',
    '# of Trip Samples': 'sample_size',       # 2025 format
    'sample_count': 'sample_size',            # 2024 format
    'day_type_name': 'day_type',              # 2016-2023 format
}

keep_cols = ['route_id', 'direction_id', 'stop_id', 'stop_name', 'day_type',
             'boardings', 'alightings', 'sample_size', 'season', 'year']

def clean_ridership(file_name):

    df = pd.read_csv(raw_folder + '\\' + file_name, dtype=str)

    df.columns = df.columns.str.strip()                        # remove stray spaces
    df = df.rename(columns=rename_map)                         # standard names
    df = df[[c for c in keep_cols if c in df.columns]]         # keep what we need

    # Year: last 4 characters work for "2024", "Fall 2025" and "Fall 2016"
    if 'year' in df.columns:
        df['year'] = df['year'].str.strip().str[-4:].astype(int)
    else:
        df['year'] = df['season'].str.strip().str[-4:].astype(int)

    # Numbers come in as text; convert them
    for col in ['boardings', 'alightings', 'sample_size', 'direction_id']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Standard day-type labels (2025 uses short ones)
    df['day_type'] = df['day_type'].str.lower().str.strip()
    df['day_type'] = df['day_type'].replace({'wkdy': 'weekday', 'sat': 'saturday',
                                             'sun': 'sunday', 'fri': 'friday'})

    # 2025 splits Mon-Thu and Friday: weight 4/5 and 1/5, then call both weekday
    if 'friday' in df['day_type'].unique():
        mon_thu = df['day_type'] == 'weekday'
        friday  = df['day_type'] == 'friday'
        df.loc[mon_thu, ['boardings', 'alightings']] = df.loc[mon_thu, ['boardings', 'alightings']] * 0.8
        df.loc[friday,  ['boardings', 'alightings']] = df.loc[friday,  ['boardings', 'alightings']] * 0.2
        df.loc[friday, 'day_type'] = 'weekday'

    df['fare_free'] = df['route_id'].isin(fare_free_routes)
    df = df.drop(columns=['season'], errors='ignore')

    # Add up all trips at each stop: one row per route, direction, stop, day type
    # (this also merges the weighted Friday rows into weekday)
    df = (df.groupby(['year', 'route_id', 'direction_id', 'stop_id', 'stop_name', 'day_type', 'fare_free'],
                     as_index=False)
            .agg(boardings=('boardings', 'sum'),
                 alightings=('alightings', 'sum'),
                 trips_sampled=('sample_size', 'sum')))

    return df

print('Function loaded')    # confirms this cell ran

Function loaded


In [2]:
r2016 = clean_ridership('MBTA_Bus_Ridership_by_Trip_Season_Route_Line_and_Stop_Fall_2016.csv')
r2017 = clean_ridership('MBTA_Bus_Ridership_by_Trip_Season_Route_Line_and_Stop_Fall_2017.csv')
r2018 = clean_ridership('MBTA_Bus_Ridership_by_Trip_Season_Route_Line_and_Stop_Fall_2018.csv')
r2019 = clean_ridership('MBTA_Bus_Ridership_by_Trip_Season_Route_Line_and_Stop_Fall_2019.csv')
r2020 = clean_ridership('MBTA_Bus_Ridership_by_Trip_Season_Route_Line_and_Stop_Fall_2020.csv')
r2021 = clean_ridership('MBTA_Bus_Ridership_by_Trip_Season_Route_Line_and_Stop_Fall_2021.csv')
r2022 = clean_ridership('MBTA_Bus_Ridership_by_Trip_Season_Route_Line_and_Stop_Fall_2022.csv')
r2023 = clean_ridership('MBTA_Bus_Ridership_by_Trip_Season_Route_Line_and_Stop_Fall_2023.csv')
r2024 = clean_ridership('MBTA_Bus_Ridership_by_Trip_Season_Route_Line_and_Stop_Fall_2024.csv')
r2025 = clean_ridership('MBTA_Bus_Ridership_by_Trip_Season_Route_Line_and_Stop_Fall_2025.csv')

print('Loaded')

Loaded


In [3]:
rider = pd.concat([r2016, r2017, r2018, r2019, r2020, r2021, r2022, r2023, r2024, r2025],
                     ignore_index=True)

In [4]:
route_year = (rider.groupby(['route_id', 'year', 'day_type', 'fare_free'], as_index=False)['boardings']
                   .sum()
                   .rename(columns={'boardings': 'avg_daily_boardings'}))

route_year.to_csv(processed + r'\ridership_route_year.csv', index=False)
print(route_year.shape)

(3821, 5)


In [5]:
rider.to_csv(processed + r'\ridership_stops_year.csv', index=False)
print(rider.shape)

(287428, 10)
